# Quickstart

## Your First Kafi Streams Topology

As in Kafka Streams, in Kafi Streams, a processing pipeline is called *topology*.

A Kafi Streams topology is a directed acyclic graph of operators starting with arbitrary many *sources* and ending with arbitrary many *sinks* (typically corresponding to Kafka topics).

To clear things up, here is a concrete example, displayed in the traditional Kafka Streams-like way (see https://zz85.github.io/kafka-streams-viz/):

```mermaid
graph TD
25aa7736-238a-40f0-8054-d9c6f986ee3b[source_clicks] --> a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op]
fcb627f1-8b99-408c-941a-130bfbf828c4[map_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op] --> 56b14bac-535b-4bfa-81ae-6347726eb5f9[sink_joined]
14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op] --> 14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op]
2c812b6a-4b42-4293-a143-b36e797d0a84[source_customers] --> fcb627f1-8b99-408c-941a-130bfbf828c4[map_op]
```

There are two sources (`source_clicks` and `source_customers`) at the top. Each source goes through a `map` operaotor. The clicks, in addition, go through a `filter` operator. Then, both sides are joined by the `join_equi` operator and end up in the sink (`sink_joined`).



## How Does the Data Look Like?

We assume that `clicks` is a source of Kafka messages like this:

In [ ]:
click_m_dict = {
    "key": None,
    "value": {"customer_id": "4711",
              "view_time": 200},
    "partition": 2,
    "offset": 23,
    "timestamp": 1609457201000,
    "headers": None
}

...and `customers` is a source of Kafka messages like this:

In [ ]:
customer_m_dict = {
    "key": "4711",
    "value": {"id": "4711",
              "name": "Sallyann Jupp"},
    "partition": 0,
    "offset": 67,
    "timestamp": 1609457001000,
    "headers": None
}

The aim of the topology is to join `clicks` with `customers` to enrich the `customer_id` in the `clicks` with the `name` of the customer from the `customers`. An example output message looks like this (only the `value` field of the Kafka message is relevant here):

```python
{"value": {"customer_id": "4711",
           "view_time": 100,
           "name": "Sallyann Jupp"}
}
```

In [15]:
import sys
sys.path.insert(1, "..")

import importlib
import kafi.streams.topologynode
import kafi.streams.streams
importlib.reload(kafi.streams.topologynode)
importlib.reload(kafi.streams.streams)

from kafi.streams.streams import Streams

import logging

logging.basicConfig(level=logging.INFO)

# "Connect" to Kafi's Kafka emulation on local disk
# from kafi.fs.local.local import Local
# c = Local({"local": {"root.dir": "/tmp"}})

# Connect to real local Kafka instaed
from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

#

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"
#
click_tn = (
    Streams.source(c, click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    .filter(lambda r: r["view_time"] < 100)
)
#
customer_tn = (
    Streams.source(c, customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)
#
sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "view_time": l["view_time"],
            "name": r["name"]}})
    .sink(c, sink_str)
)
#
built_tn = Streams.build(sink_tn)


In [16]:
print(built_tn.mermaid())

```mermaid
graph TD
25aa7736-238a-40f0-8054-d9c6f986ee3b[source_clicks] --> a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op]
fcb627f1-8b99-408c-941a-130bfbf828c4[map_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op] --> 56b14bac-535b-4bfa-81ae-6347726eb5f9[sink_joined]
14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op] --> 14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op]
2c812b6a-4b42-4293-a143-b36e797d0a84[source_customers] --> fcb627f1-8b99-408c-941a-130bfbf828c4[map_op]
```


## Setup

For simplicity, we use Kafi's emulated Kafka on your local disk (replace the `root.dir` with your preferred directory where you'd like Kafi's emulated Kafka to put its files).

The interesting part is the topology specification `tn=...`. It consists of five steps (excluding the `peek` calls for debugging, you can ignore them for the time being):

1. We specify the source. Unlike Kafka Streams, each source can be on any Kafka cluster (here: `c`). The source topic name is `orders` (=`source_str`)
2. We are only interested in the `value` of the messages - hence, we just select that from the Kafka messages. In principle, you can access any field from the Kafka messages, not just the key and value but also the partition, offset, timestamp and the headers.
3. We group the messages by their `customer_id` and aggregate the respective orders and products. The "projection" of the aggregation is a record including the `customer_id`, `orders` and `product_ids`:
    * orders are just counted in `orders`
    * the product IDs are accumulated in the list `product_ids`
4. We recreate the Kafka message layout - putting the `customer_id` into the key and the aggregated record into the value.
5. We specify the sink. Again, unlike Kafka Streams, each sink can be on any Kafka cluster (here: `c` again). The sink topic name is `orders_aggregated` (=`sink_str`)

That's it. Ready to rumble.

Let's first create a generator for randomly generating orders:

In [ ]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        message_dict = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


And then, in the next step:
1. "Build" the topology specified above to make it ready for running.
2. (Re-)create the source and sink topics.
3. Start Streams as a Python task.

...and:

4. Get the generator.
5. Set up the producer.
6. 10 times: Generate a message and produce it to Kafka.
7. Close the producer.

Since we added `peek` calls after each step, you'll see the outputs of the individual steps now as Streams picks up the produced messages.

In [ ]:
# 1. Build the topology.
built_tn = Streams.build(tn)
# 2. (Re-)create the source and sink topics.
c.recreate(source_str)
c.recreate(sink_str)
# 3. Start Streams on the built topology as a Python task.
stop = Streams.start_streams(built_tn)

#

# 4. Get the generator.
gen = OrderGenerator()
# 5. Set up the producer.
pr = c.producer(source_str)
# 6. Loop (10 times): Generate a message + produce it to Kafka.
for _ in range(10):
    message_dict = gen.generate()
    pr.produce(message_dict["value"], key=message_dict["key"])
# 7. Close the producer.
pr.close()

#

print("Streams started...")



We can now stop our Streams task...


In [ ]:
stop()

...and then print out the sink topic:

In [ ]:
c.cat(sink_str)

That's it. Congratulations - that was your first Streams topology in action :-)

That's how stream processing looks like 2026. Natural.